In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
import os

sys.path.append(os.path.abspath("../src"))
from denoising.eval.NoiseCorrelations import *
from denoising.figures.NoiseCorrelations_Fig8 import *

## Spatial and k-Space Normalization

We define the field of view (FOV) to be identical for both Cartesian (uniform phase encoding)
and concentric ring trajectory (CRT) sampling and normalize it to

$$
\mathrm{FOV} = 1 .
$$

The reconstructed image is interpreted as one spatial period of a periodic object.
With \(N\) pixels per spatial dimension, the spatial sampling interval is

$$
\Delta x = \frac{\mathrm{FOV}}{N} = \frac{1}{N},
$$

resulting in an \(N \times N\) discrete image grid.

The spatial sample locations are given by

$$
\mathbf{x}_n = \frac{1}{N}(n_x,n_y),
\qquad
n_x,n_y \in \left\{-\tfrac{N}{2},\dots,\tfrac{N}{2}-1\right\},
$$

which correspond to spatial coordinates

$$
\mathbf{x}_n \in (-0.5,\,0.5)\times(-0.5,\,0.5).
$$

---

## k-Space Support and Nyquist Conditions

Sampling the image on a discrete spatial grid with spacing \(\Delta x\) imposes a
Nyquist-limited maximum spatial frequency

$$
k_{\max} = \frac{1}{2\,\Delta x} = \frac{N}{2},
$$

which determines the achievable spatial resolution and is identical for Cartesian and CRT
sampling.

Conversely, discretizing k-space introduces a second, dual Nyquist condition. If k-space
samples are spaced by \(\Delta k\), the reconstructed image becomes periodic with period

$$
\mathrm{FOV} = \frac{1}{\Delta k}.
$$

To avoid spatial aliasing (wrap-around of the object within the FOV), the k-space sampling
interval must therefore satisfy

$$
\boxed{
\Delta k \le \frac{1}{\mathrm{FOV}}
}
$$

which, for the chosen normalization \(\mathrm{FOV} = 1\), reduces to

$$
\Delta k \le 1.
$$

In the Cartesian case, k-space samples are located at integer-valued Fourier modes

$$
\mathbf{k} = (k_x,k_y),
\qquad
k_x,k_y \in \left\{-\tfrac{N}{2},\dots,\tfrac{N}{2}-1\right\},
$$

corresponding to a k-space sampling interval \(\Delta k = 1\), which exactly satisfies the
Nyquist condition.

For CRT sampling, k-space is sampled along concentric rings with finite radial and angular
spacing. The local k-space sampling density must therefore be chosen such that the effective
radial and azimuthal sample spacings satisfy the same Nyquist condition in order to avoid
spatial aliasing.

The Fourier reconstruction kernel is evaluated as

$$
e^{i 2\pi \mathbf{k}\cdot \mathbf{x}} .
$$



# Compute Cartesian backward operator

# Compute CRT backward operator

In [ ]:
N = 22

# --------------------
# Cartesian reference
# --------------------
kx_c, ky_c = cartesian_k_grid_fov1(N)
B_cart = backward_operator_from_k(N, kx_c, ky_c, w=None, alpha="trace")
C_cart = B_cart @ B_cart.conj().T


# --------------------
# CRT WITHOUT DC
# --------------------
kx, ky, w = crt_2d_rings_const_theta(
    N,
    oversamp_factor=1.0,
    kmax=N/2,
    dk_target=1.0,
    include_dc=False,
    verbose=True
)

B_crt = backward_operator_from_k(N, kx, ky, w=w, alpha="trace")
C_crt = B_crt @ B_crt.conj().T

In [ ]:
plot_covariances(C_cart, C_crt)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# correlation matrix
R_crt = cov_to_corr(C_crt)

plt.figure(figsize=(4,4))
plot_corr_to_center(
    R_crt,
    N,
    title="Spatial noise correlation relative to central voxel\n(CRT, $N=22$)",
    vmax=0.3   # see discussion below
)

plt.tight_layout()
plt.savefig("SavedGraphics/Fig_S1_noise_correlation.pdf", bbox_inches="tight")
plt.show()

